Write your solution in here.

- Your code has to reference all data files using **relative** paths so that they can be loaded on other computers without modifications.
- All data files need to be stored in the `data/` folder.

For example:

In [267]:
import pandas as pd

# Define the path to the data directory
DATA_PATH = './data'

# Read the CSV using a relative path
df = pd.read_csv(f'{DATA_PATH}/sce_extract_2015.csv')

# Term Paper 2 - Expectations about Macroeconomic and Financial Variables

## Part 1 - Data Preprocessing (SCE)

In [268]:
import pandas as pd
import numpy as np
import glob

DATA_PATH = './data'

# Variables asked only at the first interview → need forward-filling
COLS_FIRST_WAVE = [
    'owner', 'health', 'age_init',
    'num_lit_q1_correct', 'num_lit_q2_correct', 'num_lit_q3_correct',
    'num_lit_q5_correct', 'num_lit_q6_correct',
    'num_lit_q8_correct', 'num_lit_q9_correct',
]

COLS_EXPECTATIONS = ['infl_1y', 'house_price_change', 'prob_unrate_up', 'prob_stocks_up']

COLS_INDIVIDUAL = [
    'female', 'hispanic', 'black', 'educ',
    'hh_inc_bin_rank', 'working', 'couple', 'num_kids',
    'financial_past_12m', 'financial_12m', 'take_fin_risk'
]

COLS_META = ['userid', 'wid', 'date', 'tenure', 'weight']
ALL_COLS  = COLS_META + COLS_EXPECTATIONS + COLS_FIRST_WAVE + COLS_INDIVIDUAL

files = sorted(glob.glob(f'{DATA_PATH}/sce_extract_*.csv'))

df = pd.concat(
    [pd.read_csv(f, usecols=lambda c: c in ALL_COLS, parse_dates=['date']) for f in files],
    ignore_index=True
).sort_values(['userid', 'date']).reset_index(drop=True)

n_obs_init   = len(df)
n_ind_init   = df['userid'].nunique()
n_waves_init = df['wid'].nunique()

first_date_init = df['date'].min().date()
last_date_init  = df['date'].max().date()

print("=== Initial sample ===")
print(f"  Observations : {n_obs_init:,}")
print(f"  Individuals  : {n_ind_init:,}")
print(f"  Survey waves : {n_waves_init}")
print(f"  Date range   : {first_date_init} → {last_date_init}")

=== Initial sample ===
  Observations : 180,268
  Individuals  : 23,886
  Survey waves : 143
  Date range   : 2013-06-01 → 2025-05-01


In [269]:
# Forward-fill within each individual (only fills NaNs after the first observed value)
df[COLS_FIRST_WAVE] = (
    df.groupby('userid')[COLS_FIRST_WAVE]
    .transform(lambda s: s.ffill())
)

In [270]:
# Drop observations from 2025 due to Trump-volatility
mask_2025 = df['date'].dt.year == 2025
n_dropped_2025 = mask_2025.sum()
df = df[~mask_2025].copy()
print(f"Step 3 — Dropped {n_dropped_2025:,} observations from 2025")

Step 3 — Dropped 4,167 observations from 2025


In [ ]:
# Final set of columns to use in the analysis, before NA analysis (Don't print this in the final solution, just for reference)
ANALYSIS_COLS = COLS_EXPECTATIONS + COLS_FIRST_WAVE + COLS_INDIVIDUAL

df[ANALYSIS_COLS].isna().agg(['sum', 'mean']).T.rename(columns={'sum': 'Count NA', 'mean': 'Share NA'}).sort_values('Share NA', ascending=False)

,Count NA,Share NA
num_lit_q9_correct,36718.0,0.208505
num_lit_q8_correct,36396.0,0.206677
take_fin_risk,36355.0,0.206444
health,36342.0,0.206370
couple,12992.0,0.073776
hh_inc_bin_rank,1631.0,0.009262
num_lit_q6_correct,1053.0,0.005980
prob_stocks_up,960.0,0.005451
educ,742.0,0.004213
infl_1y,682.0,0.003873


In [272]:
# Decide which columns to keep based on the NA analysis and substantive importance
COLS_FIRST_WAVE_USED = [
    'owner', 'age_init',
    'num_lit_q1_correct', 'num_lit_q2_correct', 'num_lit_q3_correct',
    'num_lit_q5_correct', 'num_lit_q6_correct'
    # maybe keep 'health' if you think it’s substantively important?
]

COLS_INDIVIDUAL_USED = [
    'female', 'black', 'hispanic', 'educ',
    'hh_inc_bin_rank', 'working', 'couple',
    'num_kids',
    'financial_past_12m', 'financial_12m'
    # consider whether to keep 'take_fin_risk'
]

# Final set of columns to use in the analysis
ANALYSIS_COLS = COLS_EXPECTATIONS + COLS_FIRST_WAVE_USED + COLS_INDIVIDUAL_USED

In [273]:


n_before = len(df)
df = df.dropna(subset=ANALYSIS_COLS).reset_index(drop=True)
print(f"Step 4 — Dropped {n_before - len(df):,} observations with missing values")

Step 4 — Dropped 16,966 observations with missing values


In [274]:
for col in COLS_EXPECTATIONS:
    p1  = df[col].quantile(0.01)
    p99 = df[col].quantile(0.99)
    n_before = len(df)
    df = df[(df[col] > p1) & (df[col] < p99)]
    print(f"  {col}: P1={p1:.2f}, P99={p99:.2f} → dropped {n_before - len(df):,} obs")

df = df.reset_index(drop=True)

  infl_1y: P1=-30.00, P99=60.00 → dropped 3,400 obs
  house_price_change: P1=-20.00, P99=40.00 → dropped 4,406 obs
  prob_unrate_up: P1=0.00, P99=99.00 → dropped 4,281 obs
  prob_stocks_up: P1=1.00, P99=90.00 → dropped 4,781 obs


In [275]:
df['optimist_unrate']      = (df['prob_unrate_up'] < 50).astype(int)
df['optimist_stocks']      = (df['prob_stocks_up'] > 50).astype(int)
df['optimist_house_price'] = (df['house_price_change'] > 0).astype(int)

summary = pd.DataFrame({
    'Sample'       : ['Initial', 'Final'],
    'Observations' : [n_obs_init, len(df)],
    'Individuals'  : [n_ind_init, df['userid'].nunique()],
    'Survey waves' : [n_waves_init, df['wid'].nunique()],
    'First date'   : [first_date_init, df['date'].min().date()],
    'Last date'    : [last_date_init, df['date'].max().date()],
})
display(summary)

,Sample,Observations,Individuals,Survey waves,First date,Last date
0,Initial,180268,23886,143,2013-06-01,2025-05-01
1,Final,142267,21628,139,2013-06-01,2024-12-31


## Part 2 - Data Preprocessing (Macro/Finance)

NOT INCLUDED IN FINAL CODE:

    - Yahoo finance data uploaded using python code

    - Yahoo finance data files cleaned to data-price format, saved as .csv

    - FRED macro data downloaded manually (required package not available in python environment)

Data is downloaded, partially cleaned and loaded in notebook, but merging of dataframes is currently a mess

In [276]:
# Yahoo finance data: Previously downloaded in Python

DATA_PATH = "./data"

wti   = pd.read_csv(f'{DATA_PATH}/wti_crude_oil.csv', parse_dates=['date'], index_col='date')
sp500 = pd.read_csv(f'{DATA_PATH}/sp500.csv', parse_dates=['date'], index_col='date')

display(wti.head())
display(sp500.head())

,wti_crude_oil
date,
2010-01-04,81.510002
2010-01-05,81.769997
2010-01-06,83.180000
2010-01-07,82.660004
2010-01-08,82.750000


,sp500
date,
2010-01-04,1132.989990
2010-01-05,1136.520020
2010-01-06,1137.140015
2010-01-07,1141.689941
2010-01-08,1144.979980


In [277]:
# Monthly finance data
# Month-end levels
wti_m   = wti.resample("ME").last()
sp500_m = sp500.resample("ME").last()

# Optional: monthly percentage returns (for later features)
wti_m["wti_ret"]    = wti_m["wti_crude_oil"].pct_change()
sp500_m["sp500_ret"] = sp500_m["sp500"].pct_change()

# Limit to the date range of the survey data
start_date = df['date'].min()
date_range = pd.date_range(start=start_date, end=df['date'].max(), freq='ME')
wti_m   = wti_m.reindex(date_range)
sp500_m = sp500_m.reindex(date_range)

# Apply timing rule: at month t we only use info up to end of month t-1 → shift by 1 month
wti_m   = wti_m.shift(1)
sp500_m = sp500_m.shift(1)

display(wti_m.head())
display(sp500_m.head())

,wti_crude_oil,wti_ret
2013-06-30,NaN,NaN
2013-07-31,96.559998,0.049908
2013-08-31,105.029999,0.087717
2013-09-30,107.650002,0.024945
2013-10-31,102.330002,-0.049419


,sp500,sp500_ret
2013-06-30,NaN,NaN
2013-07-31,1606.280029,-0.014999
2013-08-31,1685.729980,0.049462
2013-09-30,1632.969971,-0.031298
2013-10-31,1681.550049,0.029750


In [278]:
# FRED data: Downloaded manually

DATA_PATH = "./data"

macro_files = {
    "cpi":               "CPIAUCSL.csv",
    "house_price_idx":   "USSTHPI.csv",
    "federal_funds":     "FEDFUNDS.csv",
    "unemployment":      "UNRATE.csv",
    "real_gdp":          "GDPC1.csv",
    "usd_index":         "DTWEXBGS.csv",
    "mortgage_30y":      "MORTGAGE30US.csv",
}

def load_fred_series(fname, colname):
    """Load a FRED CSV (date in first col, value in second)."""
    df = pd.read_csv(f"{DATA_PATH}/{fname}", parse_dates=[0])
    df = df.rename(columns={df.columns[0]: "date", df.columns[1]: colname})
    return df.set_index("date").sort_index()

# 1) Load raw series
macro_raw = {
    name: load_fred_series(fname, name)
    for name, fname in macro_files.items()
}

# 2) Convert everything to MONTHLY frequency
macro_m = {}

# CPI (index, monthly) -> keep last obs in month, maybe YoY inflation
cpi = macro_raw["cpi"].resample("ME").last()
cpi["cpi_yoy"] = cpi["cpi"].pct_change(12) * 100
macro_m["cpi"] = cpi

# House price index (often quarterly, but resampled to monthly & ffilled)
hpi = macro_raw["house_price_idx"].resample("ME").last().ffill()
macro_m["house_price_idx"] = hpi

# Federal funds rate (daily/weekly) -> monthly average
ffr = macro_raw["federal_funds"].resample("ME").mean()
macro_m["federal_funds"] = ffr

# Unemployment rate (monthly) -> last obs each month
unrate = macro_raw["unemployment"].resample("ME").last()
macro_m["unemployment"] = unrate

# Real GDP (quarterly) -> monthly by forward-fill within quarter
gdp = macro_raw["real_gdp"].resample("ME").last().ffill()
macro_m["real_gdp"] = gdp

# Dollar index (daily) -> month-end value
usd = macro_raw["usd_index"].resample("ME").last()
macro_m["usd_index"] = usd

# 30y mortgage rate (daily/weekly) -> monthly average
mort = macro_raw["mortgage_30y"].resample("ME").mean()
macro_m["mortgage_30y"] = mort

# 3) Apply timing rule: at month t we only use info up to end of month t-1
for name in macro_m:
    macro_m[name] = macro_m[name].shift(1)

# 4) Combine into single monthly macro panel
macro_all = pd.concat(macro_m.values(), axis=1).sort_index()
# 5) Keep only dates that overlap with survey data
macro_all = macro_all[(macro_all.index >= df['date'].min()) & (macro_all.index <= df['date'].max())]
display(macro_all.head())

/var/folders/5b/2gvzw65d5_j3rxrrg64d8fh40000gn/T/ipykernel_35971/2043244844.py:32: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  cpi["cpi_yoy"] = cpi["cpi"].pct_change(12) * 100


,cpi,cpi_yoy,house_price_idx,federal_funds,unemployment,real_gdp,usd_index,mortgage_30y
date,,,,,,,,
2013-06-30,231.893,1.390389,321.03,0.11,7.5,17709.671,NaN,3.536
2013-07-31,232.445,1.715794,321.03,0.09,7.5,17709.671,NaN,4.070
2013-08-31,232.900,1.885472,325.58,0.09,7.3,17860.450,NaN,4.370
2013-09-30,233.456,1.538809,325.58,0.08,7.2,17860.450,NaN,4.456
2013-10-31,233.544,1.094734,325.58,0.08,7.2,17860.450,NaN,4.490


In [279]:
macro_all['usd_index'].isna().sum()

np.int64(95)

In [280]:
# Merge macro and finance data
macro_all = macro_all.join(wti_m, how="outer")
macro_all = macro_all.join(sp500_m, how="outer")

macro_all = macro_all.sort_index()
display(macro_all.head())

# Merge macro and finance data to SCE
sce_data = df.join(macro_all, how="outer")
display(sce_data.head())

,cpi,cpi_yoy,house_price_idx,federal_funds,unemployment,real_gdp,usd_index,mortgage_30y,wti_crude_oil,wti_ret,sp500,sp500_ret
date,,,,,,,,,,,,
2013-06-30,231.893,1.390389,321.03,0.11,7.5,17709.671,NaN,3.536,NaN,NaN,NaN,NaN
2013-07-31,232.445,1.715794,321.03,0.09,7.5,17709.671,NaN,4.070,96.559998,0.049908,1606.280029,-0.014999
2013-08-31,232.900,1.885472,325.58,0.09,7.3,17860.450,NaN,4.370,105.029999,0.087717,1685.729980,0.049462
2013-09-30,233.456,1.538809,325.58,0.08,7.2,17860.450,NaN,4.456,107.650002,0.024945,1632.969971,-0.031298
2013-10-31,233.544,1.094734,325.58,0.08,7.2,17860.450,NaN,4.490,102.330002,-0.049419,1681.550049,0.029750


,userid,wid,tenure,weight,date,financial_past_12m,financial_12m,prob_unrate_up,prob_stocks_up,infl_1y,...,house_price_idx,federal_funds,unemployment,real_gdp,usd_index,mortgage_30y,wti_crude_oil,wti_ret,sp500,sp500_ret
0,70000339.0,201306.0,7.0,2.93,2013-06-05,2.0,3.0,20.0,30.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,70000341.0,201306.0,7.0,2.61,2013-06-05,4.0,4.0,40.0,60.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,70003183.0,201306.0,7.0,2.83,2013-06-11,3.0,3.0,30.0,10.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,70003189.0,201306.0,6.0,1.88,2013-06-02,4.0,4.0,6.0,43.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,70003202.0,201306.0,7.0,3.79,2013-06-21,4.0,3.0,34.0,50.0,2.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


userid                     139
wid                        139
tenure                     139
weight                     157
date                       139
financial_past_12m         139
financial_12m              139
prob_unrate_up             139
prob_stocks_up             139
infl_1y                    139
working                    139
house_price_change         139
num_lit_q1_correct         139
num_lit_q2_correct         139
num_lit_q3_correct         139
num_lit_q5_correct         139
num_lit_q6_correct         139
num_lit_q8_correct       20610
num_lit_q9_correct       20836
age_init                   139
female                     139
hispanic                   139
black                      139
educ                       139
owner                      139
health                   20629
take_fin_risk            20643
num_kids                   139
couple                     139
hh_inc_bin_rank            139
optimist_unrate            139
optimist_stocks            139
optimist